In [ ]:
import numpy as np
import pandas as pd

import seaborn as sns
import pylab as plt

from sklearn import decomposition, preprocessing, cross_decomposition

from importlib import reload
import sys
import os
sys.path.insert(1,'../../make_figures')
import plots
sys.path.insert(1,'../3_logistic_regression/')
import load_modalities

In [ ]:
def get_PCs(data,n=2):
    scaled = data.copy(deep=True)
    scale_cols = [col for col in data
             if ~(np.isin(data[col].dropna().unique(), [0, 1]).all())]
    if scale_cols:
        scaled[scale_cols] = preprocessing.StandardScaler().fit_transform(scaled[scale_cols])
    pca = decomposition.PCA(n_components=n)
    pca.fit(scaled)
    transformed = pca.transform(scaled)
    explained_var = pca.explained_variance_ratio_
    return transformed, explained_var

In [ ]:
reload(plots)
image_path = plots.set_dir()
colors = plots.set_colors()
data_path = '/scratch/c.c21013066/data/ukbiobank'

In [ ]:
# load data
data = pd.read_csv(f'{data_path}/merged_data/unaffectedNoOsteoMatchedHC.csv')

In [ ]:
data.groupby('Status')['diag_PDProdHC'].value_counts()

In [ ]:
data['diagnosis'].value_counts(dropna=False)

In [ ]:
name = 'ParkinsonDisease'
merged = data[data['diagnosis']==name]
# get modalities
covs,allfeatures,allfeatures_scale,blood,blood_scale,lifestyle,lifestyle_scale,genetics,genetics_scale,prod,prod_acc = load_modalities.load_features(f'{data_path}')

In [ ]:
allfeatures.remove('male')
allfeatures.remove('accelerometry_age')

In [ ]:
data.filter(regex='family').describe()

In [ ]:
# restrict to HC
hc = data[data['Status']=='Healthy']

In [ ]:
# get modalities
preds = np.hstack([genetics,lifestyle,blood,prod,allfeatures,covs])

labels = pd.Series(np.hstack([np.repeat('genetics',len(genetics)),np.repeat('lifestyle',len(lifestyle)),np.repeat('blood',len(blood)),
                    np.repeat('prodromal signs',len(prod)),np.repeat('accelerometry',len(allfeatures)),np.repeat('covariates',len(covs))]),index=preds)
color_map = dict(zip(labels.unique(),sns.color_palette('deep')))

In [ ]:
corr = hc[preds].corr()

In [ ]:
plots.plot_context()
g = sns.clustermap(corr.astype(float),cmap='coolwarm',center=0,col_cluster=False,row_cluster=False,
             row_colors=labels.map(color_map),col_colors=labels.map(color_map),figsize=(12,10),xticklabels=False,yticklabels=False)
# Draw the legend bar for the classes                 
for label in labels.unique():
    g.ax_col_dendrogram.bar(0, 0, color=color_map[label],
                            label=label, linewidth=0)
g.ax_col_dendrogram.legend(bbox_to_anchor=(1,1), ncol=3);
plt.savefig('/scratch/c.c21013066/images/paper/ProdromalUKBB/features_modality_corr.png',bbox_inches='tight',dpi=300)
plt.savefig('/scratch/c.c21013066/images/paper/ProdromalUKBB/features_modality_corr.pdf',bbox_inches='tight',dpi=300)

In [ ]:
# find PCs for each modality
for label,cols in zip(labels.unique()[:-1],[genetics,lifestyle,blood,prod,allfeatures]):
    print(label,len(cols))
    pcs,variance = get_PCs(hc[cols],n=2)
    print(variance)
    print(pcs.shape)
    hc[f'{label}_1'] = pcs[:,0]
    try:
        hc[f'{label}_2'] = pcs[:,1]
    except:
        pass

In [ ]:
corr = hc.iloc[:,-10:].corr()
plots.plot_context()
g = sns.clustermap(corr.astype(float),cmap='coolwarm',center=0,figsize=(12,12),xticklabels=True,yticklabels=True,row_cluster=False)